# fretboard: package tour

A quick walk through what the `fretboard` package can do so far: notes, scales and chords. Fretboard visualisation is coming next.

In [1]:
import fretboard
from fretboard import Chord, Scale, note_name, parse_note

## Notes

Internally a note is a *pitch class*: an integer 0–11 with C = 0. `parse_note` accepts any case, sharps and flats; `note_name` converts back (always using sharps).

In [2]:
fretboard.NOTE_NAMES

('C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B')

In [3]:
{name: parse_note(name) for name in ["C", "c#", "Db", "A", "Bb", "B#", "Cb"]}

{'C': 0, 'c#': 1, 'Db': 1, 'A': 9, 'Bb': 10, 'B#': 0, 'Cb': 11}

In [4]:
[note_name(p) for p in (0, 4, 7, 13, -1)]  # wraps around the octave

['C', 'E', 'G', 'C#', 'B']

## Scales

A scale is a root plus semitone intervals above it. Build one by name with `Scale.from_name`.

In [5]:
sorted(fretboard.SCALE_INTERVALS)

['blues',
 'harmonic_minor',
 'major',
 'major_pentatonic',
 'melodic_minor',
 'minor',
 'minor_pentatonic']

In [6]:
a_major = Scale.from_name("A", "major")
a_major

Scale(root=9, intervals=(0, 2, 4, 5, 7, 9, 11))

In [7]:
a_major.notes, a_major.pitch_classes, len(a_major)

(('A', 'B', 'C#', 'D', 'E', 'F#', 'G#'), (9, 11, 1, 2, 4, 6, 8), 7)

Every scale in the table, rooted on A:

In [8]:
for name in fretboard.SCALE_INTERVALS:
    print(f"{name:<18}", " ".join(Scale.from_name("A", name).notes))

major              A B C# D E F# G#
minor              A B C D E F G
harmonic_minor     A B C D E F G#
melodic_minor      A B C D E F# G#
major_pentatonic   A B C# E F#
minor_pentatonic   A C D E G
blues              A C D D# E G


Flat roots work too, though output is spelled with sharps:

In [9]:
Scale.from_name("Bb", "major").notes

('A#', 'C', 'D', 'D#', 'F', 'G', 'A')

## Chords

Chords work the same way as scales.

In [10]:
sorted(fretboard.CHORD_INTERVALS)

['7',
 'aug',
 'dim',
 'dim7',
 'm7b5',
 'maj',
 'maj7',
 'min',
 'min7',
 'sus2',
 'sus4']

In [11]:
Chord.from_name("C", "maj7").notes

('C', 'E', 'G', 'B')

In [12]:
for quality in fretboard.CHORD_INTERVALS:
    chord = Chord.from_name("G", quality)
    print(f"{chord.name:<8}", " ".join(chord.notes))

G maj    G B D
G min    G A# D
G dim    G A# C#
G aug    G B D#
G sus2   G A D
G sus4   G C D
G 7      G B D F
G maj7   G B D F#
G min7   G A# D F
G m7b5   G A# C# F
G dim7   G A# C# E


### Identifying chords from notes

`Chord.from_notes` takes note names or pitch classes. The first note is the root; order and doubled notes don't matter.

In [13]:
for notes in (["C", "E", "G"], ["a", "c", "e"], [7, 11, 2, 5], ["E", "G", "B", "D", "E"]):
    print(notes, "->", Chord.from_notes(notes))

['C', 'E', 'G'] -> C maj
['a', 'c', 'e'] -> A min
[7, 11, 2, 5] -> G 7
['E', 'G', 'B', 'D', 'E'] -> E min7


Note sets that aren't in the chord table have no quality:

In [14]:
cluster = Chord.from_notes(["C", "D", "E"])
cluster.quality, cluster.name

(None, 'C (C D E)')

## Putting it together: the diatonic triads of a key

Stack thirds on each degree of a scale and identify the resulting chord.

In [15]:
def diatonic_triads(scale):
    degrees = scale.pitch_classes
    n = len(degrees)
    return [Chord.from_notes([degrees[i], degrees[(i + 2) % n], degrees[(i + 4) % n]])
            for i in range(n)]

for numeral, chord in zip(["I", "ii", "iii", "IV", "V", "vi", "vii°"],
                          diatonic_triads(Scale.from_name("C", "major"))):
    print(f"{numeral:<5}", chord)

I     C maj
ii    D min
iii   E min
IV    F maj
V     G maj
vi    A min
vii°  B dim
